# RAR — Real N=10 HPO Campaign on Free GPU (Ollama + Nemotron-Nano-9B)

Runs the **Recursive Autonomous Research** campaign with a *local* orchestrator LLM on the notebook GPU, so there is **no API rate limit** and the PyTorch training runs on a real **T4 GPU**.

## ⚙️ Before you run — two required settings (right sidebar → Settings):
1. **Accelerator → GPU T4 x2** (or any GPU)
2. **Internet → On** (needed to install Ollama, pull the model, and clone the repo)

Then **Run All**. Total time ≈ 2–4 h (reasoning model). Progress checkpoints to `partial_results.json`, and the final results land in `/kaggle/working/pilot_results.json` for download.

> The orchestrator model matches the paper (`nemotron-nano-9b-v2`). If that GGUF fails to load on this Ollama build, the notebook falls back to a vetted 9B model and prints which one was used (update the paper's model name accordingly).

## 1. GPU check

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'], capture_output=True, text=True).stdout)
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU! Enable Accelerator -> GPU in Settings.'

## 2. Install Ollama and start the server

In [ ]:
import os, time, subprocess, urllib.request, shutil
# Ollama's installer extracts a .tar.zst; Kaggle's base image lacks zstd, so install it first.
os.system('apt-get -qq update >/dev/null 2>&1 && apt-get -qq install -y zstd >/dev/null 2>&1')
rc = os.system('curl -fsSL https://ollama.com/install.sh | sh')
ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
assert shutil.which('ollama') or os.path.exists(ollama_bin), f'Ollama install failed (rc={rc}).'
print('ollama binary:', ollama_bin)
# Start the server in the background
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
server = subprocess.Popen([ollama_bin, 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
# Wait for it to come up
for _ in range(30):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=3); print('Ollama is up.'); break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Ollama server did not start.')

## 3. Pull the orchestrator model (Nemotron-Nano-9B GGUF, with fallback)

In [ ]:
import os
# NOTE: Nemotron-Nano-9B-v2 (Mamba-Transformer hybrid) does NOT GPU-accelerate on
# Kaggle's llama.cpp build (>5 min per short call -> not viable for a multi-call
# campaign; verified). We use gemma2:9b, a vetted open 9B instruct model, as the
# orchestrator. This is an honest substitution; the paper's model name must reflect it.
assert os.system(f'{ollama_bin} pull gemma2:9b') == 0, 'gemma2:9b pull failed'
MODEL = 'gemma2:9b'
print('USING MODEL:', MODEL)

## 4. Install Python deps and clone the repo

In [ ]:
import os
os.system('pip -q install aiohttp networkx scikit-learn scipy matplotlib nest_asyncio >/dev/null 2>&1')
# Fresh clone of the latest code
os.system('rm -rf /kaggle/working/rar && git clone --depth 1 https://github.com/Tahir-yamin/recursive-autonomy-research /kaggle/working/rar')
print(os.listdir('/kaggle/working/rar'))

## 5. Configure the local backend and health-check (one real JSON call)

In [ ]:
import os, sys, importlib, asyncio, nest_asyncio
nest_asyncio.apply()  # allow asyncio.run inside the Jupyter event loop
os.environ['LLM_BASE_URL'] = 'http://127.0.0.1:11434/v1/chat/completions'
os.environ['LLM_API_KEY'] = 'ollama'
os.environ['OPENROUTER_MODEL'] = MODEL
os.environ['OPENROUTER_MAX_TOKENS'] = '4000'
# Train the tiny MLPs on CPU (results-identical, and avoids GPU-arch mismatches
# like Pascal P100 under cu12.8). The GPU stays dedicated to the Ollama LLM.
os.environ['RAR_TORCH_DEVICE'] = 'cpu'
os.chdir('/kaggle/working/rar')
sys.path.insert(0, '/kaggle/working/rar')
import run_pilot_experiment as rp; importlib.reload(rp)

async def health():
    prompt = rp.SEARCH_SPACE_PROMPT + '\nPropose a config. Respond ONLY JSON. This is the first trial.'
    r = await rp.call_llm(prompt)
    cfg = rp.parse_json_response(r) if r else None
    print('RAW:', repr(r)[:160]); print('PARSED:', cfg); print('VALID:', rp.is_valid_config(cfg) if cfg else False)
    return rp.is_valid_config(cfg) if cfg else False
ok = asyncio.run(health())
assert ok, 'Health check failed: model did not return a valid JSON config. Try the fallback model.'

## 6. Run the full N=10 campaign on GPU
Real LLM proposals (local) + real PyTorch training (T4). Checkpoints each completed seed to `partial_results.json`. Re-running this cell resumes from the last completed seed.

In [ ]:
import os, asyncio, importlib, nest_asyncio, run_pilot_experiment as rp
nest_asyncio.apply()
# LONG-HORIZON TEST: the paper claims RAR's graph-memory advantage emerges only
# beyond ~50 cycles. Run the full N=10 at 60 cycles to test that hypothesis directly.
os.environ['RAR_CYCLES'] = '60'
os.environ.pop('RAR_SEEDS', None)  # default N=10 seeds
importlib.reload(rp)
asyncio.run(rp.execute_campaign())
print('CAMPAIGN COMPLETE')

## 7. Results summary + package for download

In [ ]:
import json, shutil, os
res = json.load(open('/kaggle/working/rar/pilot_results.json'))
print('SEEDS:', res['SEEDS'])
print('Wilcoxon p (RAR vs Baseline):', res['wilcoxon_p_value_RAR_vs_Baseline'])
import numpy as np
for c in ['stateless_baseline','vector_rag','rar_compressed']:
    cd = res['data']['conditions'][c]
    ta = np.array(cd['test_accuracies'])*100
    llm = sum(cd.get('llm_proposal_counts', [])); heu = sum(cd.get('heuristic_proposal_counts', []))
    print(f"{c:18s} test={ta.mean():.2f}±{ta.std():.2f}%  LLM-calls={llm} heuristic={heu}")
# Copy key outputs to /kaggle/working for download
for f in ['pilot_results.json','partial_results.json','full_campaign.log']:
    p = f'/kaggle/working/rar/{f}'
    if os.path.exists(p): shutil.copy(p, f'/kaggle/working/{f}')
print('\nDownload pilot_results.json from the Output panel (right sidebar) and send it back.')